In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
from data_generator import get_spark, generate_large_table
import time

In [2]:
spark=get_spark("Case10_PersistCache")
df = generate_large_table(spark, num_rows=10_000_000)

26/04/17 13:05:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
start=time.time()
expensive_df=(
    df
    .groupBy("account_number","category")
    .agg(
        F.sum("download_bytes").alias("total_download"),
        F.count("*").alias("event_count")
    )
    .withColumn("avg_download",F.col("total_download")/F.col("event_count"))
)
metrics=expensive_df.agg(F.avg("total_download"),F.max("event_count")).collect()
top_accounts=expensive_df.orderBy(F.desc("total_download")).limit(100).collect()
expensive_df.write.mode("overwrite").parquet("/tmp/case10_bad")
print(f" No caching: {time.time() - start:.1f}s (computed 3 times!)")

26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/17 13:06:04 WARN MemoryManager: Total allocation exceeds 95.

 No caching: 19.5s (computed 3 times!)


26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/17 13:06:05 WARN MemoryManager: Total allocation exceeds 9

# GOOD: Strategic persist + unpersist

In [4]:
start=time.time()
expensive_df2=(
    df
    .groupBy("account_number", "category")
    .agg(
        F.sum("download_bytes").alias("total_download"),
        F.count("*").alias("event_count")
    )
    .withColumn("avg_download", F.col("total_download") / F.col("event_count"))
)
expensive_df2.persist(StorageLevel.MEMORY_AND_DISK)
expensive_df2.count()
metrics=expensive_df2.agg(F.avg("total_download"),F.max("event_count")).collect()
top_accounts=expensive_df2.orderBy(F.desc("total_download")).limit(100).collect()
expensive_df2.write.mode("overwrite").parquet("/tmp/case10_good")
expensive_df2.unpersist()
print(f" With caching: {time.time() - start:.1f}s (computed once, read 3 times)")

26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/17 13:06:12 WARN MemoryManager: Total allocation exceeds 95.00% 

 With caching: 8.4s (computed once, read 3 times)


26/04/17 13:06:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
